<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 4.4: A FIRRTL Transform Example
**Prev: [Common Pass Idioms](4.3_firrtl_common_idioms.ipynb)**<br>
**Next: [The RV32I Instruction Set](5.1_riscv_isa.ipynb)**<br>

This AnalyzeCircuit transform walks a `Circuit` (our AST from Modules 4.2 and 4.3), and records the number of add ops it finds, per module. Then we extend it to count the adders of a whole hierarchy.

## Setup
Please run the following. The second cell contains the IR, the parser, `serialize` and the traversal helpers that we wrote in Modules 4.2 and 4.3, once more, since notebooks do not share definitions. Nothing is new there.

If a cell fails or hangs, see [Troubleshooting](docs/troubleshooting.md).

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
// The FIRRTL library of Modules 4.2: IR, parser, serializer and traversal helpers.
// Nothing new here: just run this cell.
// ---- Expressions ----
sealed trait Expression {
  // Apply f to the child Expressions of this node (like firrtl's `map`).
  def mapExpr(f: Expression => Expression): Expression = this match {
    case DoPrim(op, args, consts) => DoPrim(op, args.map(f), consts)
    case leaf                     => leaf
  }
}
case class Reference(name: String)             extends Expression // x, io.in[0], m0.io.out
case class Literal(tpe: String, value: String) extends Expression // UInt<3>(0h5)
case class DoPrim(op: String, args: Seq[Expression], consts: Seq[Int])
    extends Expression // add(a, b), bits(a, 3, 0), mux(c, a, b)

// ---- Statements ----
sealed trait Statement {
  def info: String // source locator, e.g. "cmd3.sc 6:19" ("" if there is none)

  // Apply f to the Expressions directly contained in this statement.
  def mapExpr(f: Expression => Expression): Statement = this match {
    case s: DefRegister      => s.copy(clock = f(s.clock))
    case s: DefRegisterReset => s.copy(clock = f(s.clock), reset = f(s.reset), init = f(s.init))
    case s: DefNode          => s.copy(value = f(s.value))
    case s: Connect          => s.copy(loc = f(s.loc), expr = f(s.expr))
    case s: Conditionally    => s.copy(pred = f(s.pred))
    case other               => other
  }
  // Apply f to the child Statements of this statement.
  def mapStmt(f: Statement => Statement): Statement = this match {
    case s: Conditionally => s.copy(conseq = s.conseq.map(f), alt = s.alt.map(f))
    case s: OtherStmt     => s.copy(children = s.children.map(f))
    case other            => other
  }
}
sealed trait IsRegister extends Statement { def name: String; def tpe: String }
case class DefRegister(info: String, name: String, tpe: String, clock: Expression) extends IsRegister
case class DefRegisterReset(
    info: String,
    name: String,
    tpe: String,
    clock: Expression,
    reset: Expression,
    init: Expression
) extends IsRegister
case class DefWire(info: String, name: String, tpe: String)                                           extends Statement
case class DefNode(info: String, name: String, value: Expression)                                     extends Statement
case class DefInstance(info: String, name: String, module: String)                                    extends Statement
case class Connect(info: String, loc: Expression, expr: Expression)                                   extends Statement
case class Conditionally(info: String, pred: Expression, conseq: Seq[Statement], alt: Seq[Statement]) extends Statement
// Everything we do not model in detail (printf, assert, mem, ...). Keeps the raw text.
case class OtherStmt(info: String, text: String, children: Seq[Statement]) extends Statement

// ---- Modules and circuits ----
case class Port(info: String, direction: String, name: String, tpe: String)
case class DefModule(info: String, name: String, isPublic: Boolean, ports: Seq[Port], body: Seq[Statement]) {
  def mapStmt(f: Statement => Statement): DefModule = copy(body = body.map(f))
}
case class Circuit(main: String, modules: Seq[DefModule], otherDecls: Seq[String] = Nil) {
  def mapModule(f: DefModule => DefModule): Circuit = copy(modules = modules.map(f))
}

// ---- Step 1: split into lines and recover the nesting from the indentation ----
case class Line(indent: Int, text: String, info: String)
case class Node(line: Line, kids: Seq[Node])

val infoRegex = """^(.*?)\s*@\[(.*)\]\s*$""".r

def toLines(text: String): List[Line] =
  text.linesIterator
    .filter(_.trim.nonEmpty)
    .map { raw =>
      val indent = raw.length - raw.dropWhile(_ == ' ').length
      raw.trim match {
        case infoRegex(code, info) => Line(indent, code, info)
        case code                  => Line(indent, code, "")
      }
    }
    .toList

// Parses the sibling lines at `indent`; a more indented line starts the children of the line before it.
def toTree(lines: List[Line], indent: Int): (Vector[Node], List[Line]) = {
  var rest     = lines
  val siblings = Vector.newBuilder[Node]
  while (rest.nonEmpty && rest.head.indent == indent) {
    val line              = rest.head
    val (kids, remaining) =
      if (rest.tail.nonEmpty && rest.tail.head.indent > indent) toTree(rest.tail, rest.tail.head.indent)
      else (Vector.empty[Node], rest.tail)
    siblings += Node(line, kids)
    rest = remaining
  }
  (siblings.result(), rest)
}

// ---- Step 2: expressions ----
// Split on commas that are not nested inside (), [], {} or <>.
def splitTop(s: String): List[String] = {
  var depth = 0
  val parts = scala.collection.mutable.ListBuffer[String]()
  val cur   = new StringBuilder
  for (c <- s) c match {
    case '(' | '[' | '{' | '<' => depth += 1; cur += c
    case ')' | ']' | '}' | '>' => depth -= 1; cur += c
    case ',' if depth == 0     => parts += cur.toString.trim; cur.clear()
    case _                     => cur += c
  }
  if (cur.nonEmpty) parts += cur.toString.trim
  parts.toList
}

val literalRegex = """^((?:UInt|SInt)(?:<\d+>)?)\((.*)\)$""".r
val primRegex    = """^([A-Za-z0-9_]+)\((.*)\)$""".r
val intRegex     = """^-?\d+$""".r

def parseExpr(text: String): Expression = text.trim match {
  case literalRegex(tpe, value) => Literal(tpe, value)
  case primRegex(op, args)      =>
    val (consts, exprs) = splitTop(args).partition(a => intRegex.matches(a))
    DoPrim(op, exprs.map(parseExpr), consts.map(_.toInt))
  case ref => Reference(ref)
}

// ---- Step 3: statements, ports, modules ----
val portRegex     = """^(input|output) (\S+) : (.*)$""".r
val regRegex      = """^reg (\S+) : (.*)$""".r
val regResetRegex = """^regreset (\S+) : (.*)$""".r
val wireRegex     = """^wire (\S+) : (.*)$""".r
val nodeRegex     = """^node (\S+) = (.*)$""".r
val instRegex     = """^inst (\S+) of (\S+)$""".r
val connectRegex  = """^connect (.*)$""".r
val whenRegex     = """^when (.*) :$""".r
val moduleRegex   = """^(public )?module (\S+) :$""".r

// An `else :` node belongs to the `when` right before it, so we convert siblings together.
def toStatements(nodes: Seq[Node]): Seq[Statement] =
  nodes.foldLeft(Vector.empty[Statement]) {
    case (done, Node(Line(_, "else :", _), kids)) =>
      done.last match {
        case w: Conditionally => done.init :+ w.copy(alt = toStatements(kids))
        case _                => sys.error("`else` without `when`")
      }
    case (done, node) => done :+ toStatement(node)
  }

def toStatement(node: Node): Statement = {
  val Node(Line(_, text, info), kids) = node
  text match {
    case regResetRegex(name, rest) =>
      val Seq(tpe, clock, reset, init) = splitTop(rest)
      DefRegisterReset(info, name, tpe, parseExpr(clock), parseExpr(reset), parseExpr(init))
    case regRegex(name, rest) =>
      val Seq(tpe, clock) = splitTop(rest)
      DefRegister(info, name, tpe, parseExpr(clock))
    case wireRegex(name, tpe)    => DefWire(info, name, tpe)
    case nodeRegex(name, value)  => DefNode(info, name, parseExpr(value))
    case instRegex(name, module) => DefInstance(info, name, module)
    case connectRegex(args)      =>
      val Seq(loc, expr) = splitTop(args)
      Connect(info, parseExpr(loc), parseExpr(expr))
    case whenRegex(pred) => Conditionally(info, parseExpr(pred), toStatements(kids), Nil)
    case other           => OtherStmt(info, other, toStatements(kids))
  }
}

def toModule(node: Node): DefModule = {
  val Node(Line(_, text, info), kids) = node
  val moduleRegex(pub, name)          = text
  val (portNodes, bodyNodes)          = kids.partition(k => portRegex.matches(k.line.text))
  val ports                           = portNodes.map { k =>
    val portRegex(dir, pname, tpe) = k.line.text
    Port(k.line.info, dir, pname, tpe)
  }
  DefModule(info, name, pub != null, ports, toStatements(bodyNodes))
}

// ---- Step 4: the whole file ----
def parseCircuit(firrtl: String): Circuit = {
  // Skip the "FIRRTL version" line and the multi-line annotation block `circuit Top :%[[ ... ]]`.
  val lines         = firrtl.linesIterator.dropWhile(!_.startsWith("circuit ")).toList
  val header        = lines.head
  val afterAnnos    = if (header.endsWith("%[[")) lines.tail.dropWhile(_.trim != "]]").tail else lines.tail
  val main          = header.stripPrefix("circuit ").takeWhile(c => c != ' ' && c != ':')
  val (children, _) = toTree(toLines(afterAnnos.mkString("\n")), 2)
  // `layer` and `extmodule` declarations are not analyzed: we keep their text to serialize it back.
  def rawLines(n: Node, depth: Int): Seq[String] =
    (("  " * depth) + n.line.text + (if (n.line.info.isEmpty) "" else s" @[${n.line.info}]")) +: n.kids.flatMap(
      rawLines(_, depth + 1)
    )
  val (modules, others) = children.partition(n => moduleRegex.matches(n.line.text))
  Circuit(main, modules.map(toModule), others.flatMap(rawLines(_, 1)))
}

def serialize(e: Expression): String = e match {
  case Reference(name)          => name
  case Literal(tpe, value)      => s"$tpe($value)"
  case DoPrim(op, args, consts) => (args.map(serialize) ++ consts).mkString(s"$op(", ", ", ")")
}

def serialize(s: Statement, indent: Int): Seq[String] = {
  val pad = "  " * indent
  val loc = (i: String) => if (i.isEmpty) "" else s" @[$i]"
  s match {
    case DefRegister(i, n, t, c)            => Seq(s"${pad}reg $n : $t, ${serialize(c)}${loc(i)}")
    case DefRegisterReset(i, n, t, c, r, v) =>
      Seq(s"${pad}regreset $n : $t, ${serialize(c)}, ${serialize(r)}, ${serialize(v)}${loc(i)}")
    case DefWire(i, n, t)          => Seq(s"${pad}wire $n : $t${loc(i)}")
    case DefNode(i, n, v)          => Seq(s"${pad}node $n = ${serialize(v)}${loc(i)}")
    case DefInstance(i, n, m)      => Seq(s"${pad}inst $n of $m${loc(i)}")
    case Connect(i, l, r)          => Seq(s"${pad}connect ${serialize(l)}, ${serialize(r)}${loc(i)}")
    case Conditionally(i, p, c, a) =>
      Seq(s"${pad}when ${serialize(p)} :${loc(i)}") ++ c.flatMap(serialize(_, indent + 1)) ++
        (if (a.isEmpty) Nil else Seq(s"${pad}else :") ++ a.flatMap(serialize(_, indent + 1)))
    case OtherStmt(i, t, kids) => Seq(s"$pad$t${loc(i)}") ++ kids.flatMap(serialize(_, indent + 1))
  }
}

def serialize(m: DefModule): Seq[String] = {
  val loc = (i: String) => if (i.isEmpty) "" else s" @[$i]"
  Seq(s"  ${if (m.isPublic) "public " else ""}module ${m.name} :${loc(m.info)}") ++
    m.ports.map(p => s"    ${p.direction} ${p.name} : ${p.tpe}${loc(p.info)}") ++ Seq("") ++
    m.body.flatMap(serialize(_, 2)) ++ Seq("")
}

def serialize(c: Circuit): String =
  (Seq(s"circuit ${c.main} :") ++ c.otherDecls ++ c.modules.flatMap(serialize)).mkString("\n")

// Pretty-prints any of our case classes as an indented tree: one node per line.
def showTree(x: Any, depth: Int = 0, label: String = ""): String = {
  val pad                     = "  " * depth + (if (label.isEmpty) "" else label + ": ")
  def isNode(v: Any): Boolean = v match {
    case _: Expression | _: Statement | _: Port | _: DefModule | _: Circuit => true
    case _                                                                  => false
  }
  x match {
    case p: Product if isNode(p) =>
      val fields        = p.productElementNames.toList.zip(p.productIterator.toList)
      val (kids, atoms) = fields.partition {
        case (_, v: Seq[_]) => v.nonEmpty && v.forall(isNode)
        case (_, v)         => isNode(v)
      }
      val head = pad + p.productPrefix + atoms.filter(_._2 != "").map { case (n, v) => s" $n=$v" }.mkString
      (head +: kids.flatMap {
        case (n, v: Seq[_]) => v.map(showTree(_, depth + 1, n))
        case (n, v)         => Seq(showTree(v, depth + 1, n))
      }).mkString("\n")
    case other => pad + other.toString
  }
}

// All statements of a body, including the ones nested inside `when`s (pre-order).
def allStatements(body: Seq[Statement]): Seq[Statement] = body.flatMap {
  case w: Conditionally => w +: (allStatements(w.conseq) ++ allStatements(w.alt))
  case o: OtherStmt     => o +: allStatements(o.children)
  case s                => Seq(s)
}

// The Expressions that appear directly in a statement (not the sub-expressions).
def directExprs(s: Statement): Seq[Expression] = s match {
  case r: DefRegister      => Seq(r.clock)
  case r: DefRegisterReset => Seq(r.clock, r.reset, r.init)
  case n: DefNode          => Seq(n.value)
  case c: Connect          => Seq(c.loc, c.expr)
  case w: Conditionally    => Seq(w.pred)
  case _                   => Nil
}

// An Expression and all its sub-expressions (pre-order).
def allExprs(e: Expression): Seq[Expression] = e match {
  case DoPrim(_, args, _) => e +: args.flatMap(allExprs)
  case leaf               => Seq(e)
}

In [ ]:
// Scala's mutable collections
import scala.collection.mutable

## Counting Adders Per Module

As described earlier, a FIRRTL circuit is represented using a tree representation:
  - A FIRRTL `Circuit` contains a sequence of `DefModule`s.
  - A `DefModule` contains a sequence of `Port`s, and a sequence of `Statement`s.
  - A `Statement` can contain other `Statement`s, or `Expression`s.
  - An `Expression` can contain other `Expression`s.

To visit all FIRRTL IR nodes in a circuit, we write functions that recursively walk down this tree. To record statistics, we will pass along a `Ledger` class and use it when we come across an add op:

In [ ]:
class Ledger {
  private var moduleName: Option[String] = None
  private val modules                    = mutable.LinkedHashSet[String]()
  private val moduleAddMap               = mutable.Map[String, Int]()
  def foundAdd(): Unit                   = moduleName match {
    case None       => sys.error("Module name not defined in Ledger!")
    case Some(name) => moduleAddMap(name) = moduleAddMap.getOrElse(name, 0) + 1
  }
  def getModuleName: String = moduleName match {
    case None       => sys.error("Module name not defined in Ledger!")
    case Some(name) => name
  }
  def setModuleName(myName: String): Unit = {
    modules += myName
    moduleName = Some(myName)
  }
  def addCount(name: String): Int = moduleAddMap.getOrElse(name, 0)
  def serialize: String           = {
    modules map { myName =>
      s"$myName => ${addCount(myName)} add ops!"
    } mkString "\n"
  }
}

### The transform framework
In the Scala FIRRTL compiler, an analysis or a transformation was a class that extended `firrtl.Transform`, got a `CircuitState` and returned a (possibly modified) one. The compiler ran the transforms that it was given between its parser and its emitter. That compiler is gone, but the idea is so simple that we can recreate it in a few lines for our AST: a `Transform` is just a function from `Circuit` to `Circuit`, and `compile` parses the text, runs the transforms in order and serializes the result.

In [ ]:
trait Transform {
  def name: String = getClass.getSimpleName
  def execute(c: Circuit): Circuit
}

def compileFIRRTL(firrtl: String, transforms: Seq[Transform]): String = {
  val parsed = parseCircuit(firrtl)
  val result = transforms.foldLeft(parsed) { (circuit, t) => t.execute(circuit) }
  serialize(result)
}

Now, let's define a Transform that walks the circuit and updates our `Ledger` whenever it comes across an adder (a `DoPrim` with op `add`).

Take some time to understand how `walkModule`, `walkStatement`, and `walkExpression` enable traversing all `DefModule`, `Statement`, and `Expression` nodes in the FIRRTL AST.

Questions to answer:
  - **Why doesn't walkModule call walkExpression?**
  - **Why does walkExpression do a post-order traversal?**
  - **Can you modify walkExpression to do a pre-order traversal of Expressions?**

In [ ]:
class AnalyzeCircuit extends Transform {
  // Called by `compileFIRRTL` to run your pass. It returns the circuit, here unchanged.
  def execute(circuit: Circuit): Circuit = {
    val ledger = new Ledger()

    // Execute the function walkModule(ledger) on every [[DefModule]] in
    // circuit, returning a new [[Circuit]] with new [[Seq]] of [[DefModule]].
    //   - "higher order functions" - using a function as an object
    //   - "function currying" - partial argument notation
    //   - "infix notation" - fancy function calling syntax
    //   - "map" - classic functional programming concept
    //   - discard the returned new [[Circuit]] because circuit is unmodified
    circuit mapModule walkModule(ledger)

    // Print our ledger
    println(ledger.serialize)

    // Return an unchanged circuit
    circuit
  }

  // Deeply visits every [[Statement]] in m.
  def walkModule(ledger: Ledger)(m: DefModule): DefModule = {
    // Set ledger to current module name
    ledger.setModuleName(m.name)

    // Execute the function walkStatement(ledger) on every [[Statement]] in m.
    //   - return the new [[DefModule]] (unmodified)
    m mapStmt walkStatement(ledger)
  }

  // Deeply visits every [[Statement]] and [[Expression]] in s.
  def walkStatement(ledger: Ledger)(s: Statement): Statement = {
    // Execute the function walkExpression(ledger) on every [[Expression]] in s.
    //   - discard the new [[Statement]] (unmodified)
    s mapExpr walkExpression(ledger)

    // Execute the function walkStatement(ledger) on every [[Statement]] in s.
    //   - return the new [[Statement]] (unmodified)
    s mapStmt walkStatement(ledger)
  }

  // Deeply visits every [[Expression]] in e.
  //   - "post-order traversal" - handle e's children [[Expression]] before e itself
  def walkExpression(ledger: Ledger)(e: Expression): Expression = {
    // Execute the function walkExpression(ledger) on every [[Expression]] in e.
    //   - return the new [[Expression]] (unmodified)
    val visited = e mapExpr walkExpression(ledger)

    visited match {
      // If e is an adder, increment our ledger
      case DoPrim("add", _, _) => ledger.foundAdd()
      case _                   =>
    }
    visited
  }
}

## Running our Transform

Now that we've defined it, let's run it on a Chisel design! First, let's define a Chisel module.

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._

In [ ]:
class AddMe(val nInputs: Int, val width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Vec(nInputs, UInt(width.W)))
    val out = Output(UInt(width.W))
  })
  io.out := io.in.reduce(_ +& _)
}

Next, let's elaborate it into FIRRTL text. (As we have seen, this is the CHIRRTL that Chisel emits; the rest of the flow is `firtool`'s job.)

In [ ]:
lazy val firrtlSerialization = getFirrtl(new AddMe(8, 4))

Finally, let's run our FIRRTL text through `compileFIRRTL`, including our custom transform. Note that it prints out the number of add ops it found! There are 8 inputs, so we expect 7 adders (`+&` is a plain `add`, whereas `+` would be an `add` followed by a `tail`; both contain exactly one `add`).

In [ ]:
println(compileFIRRTL(firrtlSerialization, Seq(new AnalyzeCircuit())))

The result is again FIRRTL text: our transform did not change the circuit, so you get your circuit back. We could now hand this text to `firtool` on the command line (`firtool --format=fir file.fir`) to continue the flow with the normal compiler. Chisel's own `ChiselStage` always starts from a Chisel generator, not from text, which is why `compileFIRRTL` in this notebook stops here.

## Counting Across a Hierarchy
Real designs consist of several modules. The ledger reports the number of adders *in the definition* of each module. Let's try on the design `Top` from Module 4.1, which instantiates `AddMe(2, 4)` and `AddMe(3, 4)`:

In [ ]:
class Top extends Module {
  val io = IO(new Bundle {
    val a = Input(UInt(4.W))
    val b = Input(UInt(4.W))
    val y = Output(UInt(5.W))
    val z = Output(UInt(5.W))
  })
  val m0 = Module(new AddMe(2, 4))
  val m1 = Module(new AddMe(3, 4))
  m0.io.in(0) := io.a
  m0.io.in(1) := io.b
  m1.io.in(0) := io.a
  m1.io.in(1) := io.b
  m1.io.in(2) := m0.io.out
  io.y        := m0.io.out
  io.z        := m1.io.out
}

lazy val topFirrtl = getFirrtl(new Top)
compileFIRRTL(topFirrtl, Seq(new AnalyzeCircuit()))
()

Each module is reported separately: `Top` contains no adder itself, it only *instantiates* modules that do. If we want to know the total number of adders in the elaborated design, we have to follow the instances. Using the instance graph of Module 4.3, the total of a module is its own count plus the totals of the modules it instantiates, *once for every instance*. Two instances of the same module count twice.

For the direct counts we do not need the ledger: with the collection-style traversal from Module 4.2 they are a few lines. Here is a more general version, which counts *any* primitive operation:

In [ ]:
def opCounts(m: DefModule): Map[String, Int] =
  allStatements(m.body)
    .flatMap(directExprs)
    .flatMap(allExprs)
    .collect { case DoPrim(op, _, _) => op }
    .groupBy(identity)
    .map { case (op, l) => op -> l.size }

lazy val topCircuit = parseCircuit(topFirrtl)
for (m <- topCircuit.modules) println(s"${m.name}: ${opCounts(m)}")

<span style="color:red">**Exercise: Total number of ops**</span><br>
Write `totalOps`, which returns, for the module `name` of `circuit`, the number of occurrences of the primitive operation `op` in that module *and all modules below it*, counting each instance. You will need `DefInstance` statements (found with `allStatements`) and `opCounts`. For `Top`, we expect 1 + 2 = 3 adds in total.

In [ ]:
def totalOps(circuit: Circuit, name: String, op: String): Int = ???

assert(totalOps(topCircuit, "AddMe", "add") == 1)
assert(totalOps(topCircuit, "AddMe_1", "add") == 2)
assert(totalOps(topCircuit, "Top", "add") == 3)

// Two instances of the same module count twice
class Twice extends Module {
  val io = IO(new Bundle { val in = Input(UInt(4.W)); val out = Output(UInt(5.W)) })
  val a  = Module(new AddMe(2, 4))
  val b  = Module(new AddMe(2, 4))
  a.io.in(0) := io.in; a.io.in(1)    := io.in
  b.io.in(0) := a.io.out; b.io.in(1) := io.in
  io.out     := b.io.out
}
lazy val twiceCircuit = parseCircuit(getFirrtl(new Twice))
println(twiceCircuit.modules.map(_.name))
assert(totalOps(twiceCircuit, "Twice", "add") == 2)
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong> (click to toggle displaying)</label>
<article>
<pre style="background-color:#f7f7f7">
def totalOps(circuit: Circuit, name: String, op: String): Int = {
  // An `extmodule` (BlackBox) is not parsed, so it is not found here: it has no operations to count
  circuit.modules.find(_.name == name).fold(0) { m =>
    val below = allStatements(m.body).collect { case DefInstance(_, _, module) => totalOps(circuit, module, op) }
    opCounts(m).getOrElse(op, 0) + below.sum
  }
}
</pre></article></div></section></div>

Both instances in `Twice` are of `AddMe(2, 4)`, which is a single module definition, so the module list shows only one `AddMe`, and the total counts it twice.

### What next?
In Chisel 7, there is no place to plug such a transform into the compilation flow itself, because that flow is implemented by `firtool`. The practical options are:
* **Analyze** the FIRRTL text as we did here, e.g. in a build tool or a test that checks properties of the generated design (the number of registers, the absence of a certain operator, the hierarchy, ...).
* **Do it while elaborating**, in Chisel, with `chisel3.reflect.DataMirror` and the `Definition`/`Instance` APIs of `chisel3.experimental.hierarchy`; this can inspect and change the design before the FIRRTL is emitted.
* **Customize `firtool`** with its command line options (`firtoolOpts` of `ChiselStage.emitSystemVerilog`), and the annotations and layers that Chisel supports, to steer the compiler.
* To implement a new compiler pass for real, write it in C++ as a CIRCT pass.

That's it for this section!

# You're done!

[Return to the top.](#top)